In [1]:
from __future__ import annotations

import copy
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights

torch.set_float32_matmul_precision("high")
plt.style.use("seaborn-v0_8-whitegrid")

DATA_ROOT = Path("Images")
ARTIFACTS_DIR = Path("artifacts_vit")
PLOTS_DIR = ARTIFACTS_DIR / "plots"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_TO_LABEL = {"Negative": 0, "Positive": 1}
LABEL_TO_CLASS = {v: k for k, v in CLASS_TO_LABEL.items()}
DISPLAY_NAMES  = {0: "Uncracked", 1: "Cracked"}

SEED                    = 42
IMAGE_SIZE              = 224
BATCH_SIZE              = 32
NUM_WORKERS             = 0
TRAIN_RATIO             = 0.70
VAL_RATIO               = 0.15
TEST_RATIO              = 0.15
NUM_EPOCHS              = 15
EARLY_STOPPING_PATIENCE = 4
MAX_GRAD_NORM           = 1.0
LEARNING_RATE           = 5e-5
WEIGHT_DECAY            = 1e-4
DROPOUT                 = 0.1
SAVE_PLOTS              = True

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Artifacts directory: {ARTIFACTS_DIR.resolve()}")

Device: cpu
Artifacts directory: /Users/maheshwarinair/Downloads/UL/artifacts_vit


## 1. Dataset index and split

In [2]:
def index_dataset(data_root: Path) -> pd.DataFrame:
    records = []
    valid_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    for class_name, label in CLASS_TO_LABEL.items():
        class_dir = data_root / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f"Missing: {class_dir}")
        for path in sorted(class_dir.iterdir()):
            if path.is_file() and path.suffix.lower() in valid_suffixes:
                records.append({"path": str(path), "class_name": class_name, "label": label})
    if not records:
        raise RuntimeError("No images found. Check DATA_ROOT.")
    return pd.DataFrame(records)


def stratified_split(frame: pd.DataFrame, train_ratio: float = TRAIN_RATIO,
                     val_ratio: float = VAL_RATIO, seed: int = SEED):
    rng = np.random.default_rng(seed)
    train_rows, val_rows, test_rows = [], [], []
    for _, group in frame.groupby("label"):
        idx = rng.permutation(len(group))
        n_train = int(len(group) * train_ratio)
        n_val   = int(len(group) * val_ratio)
        rows = group.iloc[idx]
        train_rows.append(rows.iloc[:n_train])
        val_rows.append(rows.iloc[n_train:n_train + n_val])
        test_rows.append(rows.iloc[n_train + n_val:])
    return (
        pd.concat(train_rows).reset_index(drop=True),
        pd.concat(val_rows).reset_index(drop=True),
        pd.concat(test_rows).reset_index(drop=True),
    )


dataset_df = index_dataset(DATA_ROOT)
train_df, val_df, test_df = stratified_split(dataset_df)

summary = pd.DataFrame([
    {"split": "full",  "total": len(dataset_df), "negative": (dataset_df.label==0).sum(), "positive": (dataset_df.label==1).sum()},
    {"split": "train", "total": len(train_df),   "negative": (train_df.label==0).sum(),   "positive": (train_df.label==1).sum()},
    {"split": "val",   "total": len(val_df),     "negative": (val_df.label==0).sum(),     "positive": (val_df.label==1).sum()},
    {"split": "test",  "total": len(test_df),    "negative": (test_df.label==0).sum(),    "positive": (test_df.label==1).sum()},
])
print(summary.to_string(index=False))

split  total  negative  positive
 full  24000     20000      4000
train  16800     14000      2800
  val   3600      3000       600
 test   3600      3000       600


## 2. Transforms and dataloaders

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((232, 232)),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.05, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.20, scale=(0.02, 0.08)),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class ConcreteCrackDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.frame = frame.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row["path"]) as img:
            img = img.convert("RGB")
            tensor = self.transform(img) if self.transform else transforms.ToTensor()(img)
        return tensor, torch.tensor(row["label"], dtype=torch.float32)


def build_weighted_sampler(frame: pd.DataFrame, seed: int = SEED) -> WeightedRandomSampler:
    counts = frame["label"].value_counts().sort_index()
    class_weights = {label: 1.0 / count for label, count in counts.items()}
    sample_weights = frame["label"].map(class_weights).to_numpy(dtype=np.float64)
    generator = torch.Generator()
    generator.manual_seed(seed)
    return WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
        generator=generator,
    )


def build_dataloaders(train_frame, val_frame, test_frame):
    sampler = build_weighted_sampler(train_frame)
    train_loader = DataLoader(ConcreteCrackDataset(train_frame, train_transform),
                              batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=AMP_ENABLED)
    val_loader   = DataLoader(ConcreteCrackDataset(val_frame, eval_transform),
                              batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=AMP_ENABLED)
    test_loader  = DataLoader(ConcreteCrackDataset(test_frame, eval_transform),
                              batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=AMP_ENABLED)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = build_dataloaders(train_df, val_df, test_df)
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

Train batches : 525
Val batches   : 113
Test batches  : 113


## 3. ViT-B/16 model

In [4]:
def build_vit(dropout: float = DROPOUT) -> nn.Module:
    model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
    in_features = model.heads.head.in_features
    model.heads = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, 1),
    )
    return model


model = build_vit().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"ViT-B/16 total parameters: {total_params:,}")

ViT-B/16 total parameters: 85,799,425


## 4. Metrics

In [5]:
def compute_auc(labels, scores):
    labels = np.asarray(labels, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    pos_mask = labels == 1
    neg_mask = labels == 0
    n_pos = int(pos_mask.sum())
    n_neg = int(neg_mask.sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = pd.Series(scores).rank(method="average").to_numpy()
    return float((ranks[pos_mask].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def compute_average_precision(labels, scores):
    labels = np.asarray(labels, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    total_pos = int(labels.sum())
    if total_pos == 0:
        return float("nan")
    order = np.argsort(-scores)
    sorted_labels = labels[order]
    tp = np.cumsum(sorted_labels == 1)
    fp = np.cumsum(sorted_labels == 0)
    precision = tp / np.maximum(tp + fp, 1)
    recall = tp / total_pos
    precision = np.concatenate([[1.0], precision])
    recall = np.concatenate([[0.0], recall])
    return float(np.sum((recall[1:] - recall[:-1]) * precision[1:]))


def compute_binary_metrics(labels, probs, threshold=0.5):
    labels = np.asarray(labels, dtype=np.int64)
    probs  = np.asarray(probs,  dtype=np.float64)
    preds  = (probs >= threshold).astype(np.int64)
    tp = int(((preds==1) & (labels==1)).sum())
    tn = int(((preds==0) & (labels==0)).sum())
    fp = int(((preds==1) & (labels==0)).sum())
    fn = int(((preds==0) & (labels==1)).sum())
    precision = tp / max(tp + fp, 1)
    recall    = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {
        "threshold": float(threshold),
        "accuracy":  (tp + tn) / max(len(labels), 1),
        "precision": float(precision),
        "recall":    float(recall),
        "specificity": float(specificity),
        "f1":        float(f1),
        "balanced_accuracy": (recall + specificity) / 2,
        "auroc":     compute_auc(labels, probs),
        "average_precision": compute_average_precision(labels, probs),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }


def select_best_threshold(labels, probs, metric_name="f1"):
    best_threshold, best_value, best_metrics = 0.5, float("-inf"), None
    for t in np.linspace(0.05, 0.95, 181):
        m = compute_binary_metrics(labels, probs, threshold=t)
        if m[metric_name] > best_value:
            best_value, best_threshold, best_metrics = m[metric_name], float(t), m
    return best_threshold, best_metrics

## 5. Training loop

In [6]:
def evaluate_model(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    labels_buf, probs_buf = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
                logits = model(images).squeeze(1)  # FIX: [B, 1] -> [B]
                loss   = criterion(logits, labels)
            probs = torch.sigmoid(logits)
            running_loss += float(loss.item()) * images.size(0)
            labels_buf.append(labels.cpu().numpy())
            probs_buf.append(probs.cpu().numpy())
    return running_loss / len(loader.dataset), np.concatenate(labels_buf), np.concatenate(probs_buf)


def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
            logits = model(images).squeeze(1)  # FIX: [B, 1] -> [B]
            loss   = criterion(logits, labels)
        if scaler is not None and AMP_ENABLED:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
        running_loss += float(loss.item()) * images.size(0)
    return running_loss / len(loader.dataset)


def fit_model(model, train_loader, val_loader):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
    scaler    = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED) if AMP_ENABLED else None

    best_metric, best_state = float("-inf"), copy.deepcopy(model.state_dict())
    best_threshold, best_val_metrics = 0.5, {}
    no_improve = 0
    history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        val_loss, val_labels, val_probs = evaluate_model(model, val_loader, criterion)
        val_threshold, val_metrics = select_best_threshold(val_labels, val_probs)
        scheduler.step(val_metrics["f1"])

        history.append({
            "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
            "val_f1": val_metrics["f1"], "val_recall": val_metrics["recall"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_auroc": val_metrics["auroc"], "val_threshold": val_threshold,
            "lr": optimizer.param_groups[0]["lr"], "epoch_seconds": time.time() - t0,
        })

        print(f"epoch {epoch:02d}/{NUM_EPOCHS} | "
              f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
              f"val_f1={val_metrics['f1']:.4f} | val_recall={val_metrics['recall']:.4f} | "
              f"threshold={val_threshold:.2f} | {time.time()-t0:.0f}s")

        if val_metrics["f1"] > best_metric:
            best_metric      = val_metrics["f1"]
            best_state       = copy.deepcopy(model.state_dict())
            best_threshold   = val_threshold
            best_val_metrics = val_metrics
            no_improve       = 0
        else:
            no_improve += 1

        if no_improve >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_val_metrics, best_threshold

## 6. Train

In [ ]:
model, history_df, best_val_metrics, best_threshold = fit_model(model, train_loader, val_loader)

criterion = nn.BCEWithLogitsLoss()
test_loss, test_labels, test_probs = evaluate_model(model, test_loader, criterion)
test_metrics = compute_binary_metrics(test_labels, test_probs, threshold=best_threshold)
test_metrics["loss"] = float(test_loss)

print("\nTest metrics:")
print(json.dumps(test_metrics, indent=2))

  0%|          | 0/525 [00:00<?, ?it/s]

epoch 01/15 | train_loss=0.0145 | val_loss=0.0122 | val_f1=0.9934 | val_recall=0.9967 | threshold=0.05 | 7587s


  0%|          | 0/525 [00:00<?, ?it/s]